# SprayLine Manager UI 運作說明

這份 Notebook 用簡單方式說明 `ui/manager` 首頁是怎麼運作的。  
重點不是介紹所有專案 API，而是說明 **Manager Dashboard 首頁實際會走到的資料流**。

目前 Manager UI 的主資料流可以用一句話理解：

```text
Browser ui/manager
→ GET /api/manager/dashboard
→ api/api_server.py
→ manager_date_service + manager_dashboard_service
→ DB sensor_1min / batch / alert
→ 回傳 managerView
→ 前端 render KPI、三站表格、處理建議、趨勢 Drawer
```

另外，通知功能不是走本地 FastAPI，而是直接送到外部 Apps Script：

```text
點擊「發送通知」
→ POST CONFIG.WARNING_APP_SCRIPT_URL
→ 顯示 Toast：通知已送出
```

## 1. Manager UI 的角色定位

新版 Manager Dashboard 不是工程分析頁，而是主管總覽頁。

主管進來第一眼要看到：

| 區塊 | 目的 |
|---|---|
| Header | 選日期、選小時 |
| 4 張 KPI | 快速知道今日整體狀態 |
| 三站狀態比較表 | 比較 Station 1 / Station 2 / Station 3 狀況 |
| Batch dropdown | 切換全部批號或單一批號 |
| 處理建議 | 只列出需要處理的問題 |
| 右側 Drawer | 看某站某指標的今日每小時趨勢 |

新版 UI 不再把工程細節塞在首頁，例如 detailed evidence、model explanation、engineer assignment、acknowledge 等。

## 2. Manager UI 實際用到的 API

目前 Manager 首頁的 API 範圍很小。

| 類型 | API | 用途 | UI 是否直接依賴 |
|---|---|---|---|
| 本地 API | `GET /api/manager/dashboard` | 取得 Manager 首頁所有資料 | 是，核心 API |
| 本地 API | `GET /api/manager/available-dates` | 取得可用日期與小時 | 存在，但目前首頁主要從 dashboard 的 `responseMeta` 取 |
| 外部通知 | `POST CONFIG.WARNING_APP_SCRIPT_URL` | 發送工程通知 | 是，但不是本地 FastAPI route |

最重要的是：

```text
Manager UI 首頁主要只靠 GET /api/manager/dashboard。
```

`available-dates` 雖然存在，但新版流程已經把 date/hour 選單需要的資訊放在 `/api/manager/dashboard -> responseMeta`。

## 3. `/api/manager/dashboard` 的輸入模式

`GET /api/manager/dashboard` 支援四種合法用法：

| 使用情境 | URL |
|---|---|
| 第一次載入，自動找 DB 最新資料 | `/api/manager/dashboard` |
| 只選日期，後端自動找該日期最新小時 | `/api/manager/dashboard?date=YYYY-MM-DD` |
| 指定日期與小時 | `/api/manager/dashboard?date=YYYY-MM-DD&hour=HH` |
| 指定日期、小時、批號 | `/api/manager/dashboard?date=YYYY-MM-DD&hour=HH&batch_id=BATCH-xxx` |

限制：

```text
hour 不能單獨出現。
batch_id 必須跟 date + hour 一起出現。
```

錯誤範例：

```text
/api/manager/dashboard?hour=18
/api/manager/dashboard?batch_id=BATCH-002
```

這些應該回 `400 Bad Request`。

## 4. Frontend 主要流程

前端入口主要是 `ui/manager/dashboard.js`。

簡化後流程：

```text
loadManagerData()
→ fetchRealtimeDataFromDB()
→ buildManagerDashboardRequestEndpoint()
→ fetch GET /api/manager/dashboard
→ validateManagerDashboardPayload()
→ applyManagerDatabaseResponse()
→ renderOverviewSection()
→ renderRecommendationSection()
→ renderTrendDrawer()
```

對應檔案：

| 功能 | 檔案 / Function |
|---|---|
| 組 API URL | `ui/manager/dashboard.js -> buildManagerDashboardRequestEndpoint()` |
| 打 API | `ui/manager/dashboard.js -> fetchRealtimeDataFromDB()` |
| 載入 Manager 資料 | `ui/manager/dashboard.js -> loadManagerData()` |
| 驗證 payload | `ui/manager/main/manager-api-validation.js -> validateManagerDashboardPayload()` |
| Date / Hour 選單 | `ui/manager/main/time-review.js` |
| KPI / 三站表格 | `ui/manager/main/render-overview.js` |
| 處理建議 / 發通知 | `ui/manager/main/render-recommendations.js` |

第一個 request 必須是：

```text
/api/manager/dashboard
```

不能是：

```text
/api/manager/dashboard?hour=0
```

這是之前的 bug，原因通常是把 `null` 丟進 `Number(null)`，結果變成 `0`。

## 5. Backend 主要流程

後端 route 在 `api/api_server.py`。

簡化後流程：

```text
GetManagerDashboard()
→ db_connection.get_connection()
→ _build_manager_bundle_from_database()
→ resolve_manager_date_hour()
→ get_manager_station_rows_for_hour()
→ get_manager_station_hourly_aggregates()
→ get_manager_sensor_rows_for_date()
→ get_manager_distinct_batch_count_for_date()
→ build_manager_dashboard_payload()
→ build_manager_view()
→ JSON response
```

對應檔案：

| 功能 | 檔案 / Function |
|---|---|
| API route | `api/api_server.py -> GetManagerDashboard()` |
| 建立 DB bundle | `api/api_server.py -> _build_manager_bundle_from_database()` |
| 決定 selectedDate / selectedHour | `api/manager_date_service.py -> resolve_manager_date_hour()` |
| 查指定小時三站資料 | `api/manager_date_service.py -> get_manager_station_rows_for_hour()` |
| 查今日每小時彙總 | `api/manager_date_service.py -> get_manager_station_hourly_aggregates()` |
| 查 selectedDate sensor rows | `api/manager_date_service.py -> get_manager_sensor_rows_for_date()` |
| 查今日 distinct batch count | `api/manager_date_service.py -> get_manager_distinct_batch_count_for_date()` |
| 組 dashboard payload | `api/manager_dashboard_service.py -> build_manager_dashboard_payload()` |
| 組 Manager UI 專用 payload | `api/manager_dashboard_service.py -> build_manager_view()` |

## 6. 主要資料來源

目前 Manager Dashboard 最核心的資料來源是 DB 裡的 `sensor_1min`。

| 資料 | 來源 |
|---|---|
| Date / Hour 可選範圍 | `sensor_1min.ts` |
| Station 資料 | `sensor_1min.station_id` |
| Batch 資料 | `sensor_1min.batch_id` |
| 感測值 | `sensor_1min` 的噴塗流量、空壓、噴幅、path error 等 |
| 今日 batch 數 | `COUNT(DISTINCT batch_id)` |
| alert / recommendation | 既有 alert / recommendation DB function |

特別注意：

```text
batch 數不是 sensor_1min rows 數量。
batch 數必須用 COUNT(DISTINCT batch_id)。
同一個 batch_id 不管有幾筆 sensor row，只能算 1 批。
```

## 7. `managerView` 是前端最重要的 payload

新版前端主要吃 `managerView`，不要再從一堆舊欄位自己拼資料。

`managerView` 主要欄位：

| 欄位 | 用途 |
|---|---|
| `selectionContext` | 目前選到的 date / hour / batch mode |
| `kpis` | 第一層 4 張 KPI |
| `batchSelector` | Batch dropdown |
| `stationComparison` | 三站狀態比較表 |
| `recommendations` | 處理建議 |
| `trendDrawer` | 右側趨勢 Drawer 資料 |

概念範例：

```json
{
  "managerView": {
    "selectionContext": {},
    "kpis": {},
    "batchSelector": {},
    "stationComparison": [],
    "recommendations": [],
    "trendDrawer": {}
  }
}
```

## 8. 第一層 KPI 的計算邏輯

第一層 KPI 是「今日整體」，不受 Batch dropdown 影響。

| KPI | 計算方式 |
|---|---|
| 估算 OK rate | selectedDate 當天 sensor rows 用既有 quality score 公式計算後平均 |
| 估算 NG rate | `100 - 估算 OK rate` |
| 今日產能達成率 | `今日 distinct batch_id 數 × 264 / 200000 × 100%` |
| 目前最嚴重站點 | 後端根據 issue / alert / station risk 整理 |

固定常數：

```python
MANAGER_BATCH_SIZE_PCS = 264
MANAGER_DAILY_TARGET_PCS = 200000
```

今日產能達成率最重要規則：

```text
今日 batch 數 = COUNT(DISTINCT batch_id)
今日估算產出 pcs = distinctBatchCount × 264
今日產能達成率 = 今日估算產出 pcs / 200000 × 100%
```

## 9. OK rate / NG rate 的來源

OK rate 不是人工輸入，也不是 DB 真實 OK 件數。

目前邏輯是：
def quality_score_from_formal_row
```text
從 sensor_1min 取感測資料
每一筆 sensor_1min row
→ 取噴塗流量、空壓、噴幅、路徑誤差
→ 看它偏離正常範圍多少
→ 依權重扣分
→ 得到該筆資料的 quality score / OK score
→ 多筆資料取平均
→ 得到 estimated OK rate
→ 用既有正式 quality score 公式
→ 得到 sensor-weighted estimated OK rate
```
| 欄位                  | 意義   | 對 OK rate 的影響 |
| ------------------- | ---- | ------------- |
| `paint_flow_ml_min` | 噴塗流量 | 偏離正常值會扣分      |
| `air_pressure_bar`  | 空壓   | 偏離正常值會扣分      |
| `spray_width_mm`    | 噴幅   | 偏離正常值會扣分      |
| `path_error_mm`     | 路徑誤差 | 越大扣越多         |
```text
權重比
```
| 項目                 | 最大扣分 |
| ------------------ | ---: |
| paint flow error   |   30 |
| air pressure error |   25 |
| spray width error  |   30 |
| path error         |   20 |


共用 helper：

```text
services/integrated_service/quality_score_common.py
```

原本正式邏輯來源：

```text
services/integrated_service/sprayline_integrated_service.py
_quality_score_from_formal_row(row, station_id)
```

主要欄位：

| 欄位 | 用途 |
|---|---|
| `paint_flow_ml_min` | 噴塗流量 |
| `air_pressure_bar` | 空壓 |
| `spray_width_mm` | 噴幅 |
| `path_error_mm` | 路徑誤差 |

NG rate 則固定：

```text
估算 NG rate = 100 - 估算 OK rate
```

## 10. Batch dropdown 的邏輯

Batch dropdown 放在三站狀態比較表右上角。

預設：

```text
全部批號 / 該小時累計
```

可選內容：

```text
selectedDate + selectedHour 這一小時內出現過的 DISTINCT batch_id
```

範例：

```text
全部批號 / 該小時累計
BATCH-001
BATCH-002 ⚠ NG 率偏高
BATCH-003 ⚠ 稼動率偏低
```

規則：

```text
切換 Date 或 Hour 後，Batch dropdown 自動回到「全部批號 / 該小時累計」。
```

不要保留上一個 batch，避免使用者以為還在看同一批。

## 11. 三站狀態比較表

表格欄位：

| 欄位 | 說明 |
|---|---|
| Station / Process | 站點與製程 |
| 稼動率 % | 目前選擇範圍的稼動率 |
| 預估 OK pcs | 目前選擇範圍的 OK 件數 |
| 預估 NG pcs | 目前選擇範圍的 NG 件數 |
| 主要問題 | 該站主要問題 |

第二層表格只顯示目前選擇範圍的主數字。

不要再顯示：

```text
今日累計小字
本小時小字
engineer assignment
model evidence
detailed diagnosis
```

## 12. Batch 未選 vs Batch 已選

### Batch 未選

模式：

```text
全部批號 / 該小時累計
```

每站 pcs 計算：

```text
該小時該站總 pcs = 該小時該站 DISTINCT batch_id 數 × 264
預估 OK pcs = 總 pcs × estimated OK rate / 100
預估 NG pcs = 總 pcs × estimated NG rate / 100
```

### Batch 已選

模式：

```text
指定 BATCH-xxx
```

每批固定：

```text
batch_total_pcs = 264
```

每站 pcs 計算：

```text
預估 OK pcs = 264 × 該 batch / 該 station estimated OK rate / 100
預估 NG pcs = 264 × 該 batch / 該 station estimated NG rate / 100
```

如果該 batch 尚未到某站：

```text
稼動率 —
預估 OK pcs —
預估 NG pcs —
主要問題：尚未到站
```

注意：

```text
尚未到站不是 0。
不能顯示 0 pcs。
不能顯示 No issue。
```

## 13. 右側 Drawer 趨勢圖

三站表格中，這三欄旁邊有小折線 icon：

```text
稼動率 %
預估 OK pcs
預估 NG pcs
```

只有 icon 可以點，數字本身不點。

點 icon 後右側 Drawer 打開，顯示：

| 內容 | 說明 |
|---|---|
| 標題 | Station / Process / 指標 |
| 摘要 | 依 Batch 模式不同 |
| 折線圖 | 今日 00:00 ~ 23:00 每小時趨勢 |
| Hover tooltip | 顯示精確數字 |

Batch 未選時：

```text
顯示該 Station 今日每小時「全部批號累計」趨勢
摘要：今日累計 / 本小時 / 最高小時
```

Batch 已選時：

```text
顯示該 Station 今日每小時「該 Batch」趨勢
摘要：Batch、此批總數 264 pcs、預估 OK pcs、預估 NG pcs
```

## 14. 處理建議

處理建議只顯示有問題的項目。

不要顯示：

```text
負責人
engineer assignment
assignment status
acknowledge button
model evidence
detailed diagnosis
model explanation
```

有問題時：

```text
Station 1 / Primer
問題：NG 率偏高
建議：檢查噴塗流量、空壓與噴幅設定
[發送通知]
```

沒問題時：

```text
目前無需處理事項
```

問題判斷由後端整理，前端只負責顯示。

## 15. 有問題判斷條件

符合任一條件就可以列入處理建議：
```text
581行 api\manager_dashboard_service.py 
def _pick_primary_issue
```
| 條件 | 等級 |
|---|---|
| `status = warning / alarm / critical` | 依 status |
| 估算 NG rate >= 15% | warning |
| 估算 NG rate >= 20% | alarm |
| 本小時 NG pcs >= 今日每小時平均 NG pcs × 1.5 | warning |
| 本小時 NG pcs >= 今日每小時平均 NG pcs × 2.0 | alarm |
| 今日平均稼動率 < 70% | warning |
| 今日平均稼動率 < 50% | alarm |
| 有 active alert / recommendation | 依 alert |

後端建議輸出：

```json
{
  "stationId": "Station_1",
  "stationName": "Station 1",
  "processName": "Primer",
  "level": "warning",
  "mainIssue": "NG 率偏高",
  "recommendation": "檢查噴塗流量、空壓與噴幅設定",
  "batchId": "BATCH-002"
}
```

## 16. 發送通知流程

通知不是走本地 FastAPI，而是送外部 Apps Script。

設定位置：

```text
ui/manager/main/config.js
CONFIG.WARNING_APP_SCRIPT_URL
```

前端 function：

```text
render-recommendations.js
→ sendEngineerWarningEmail()
→ buildEngineerWarningPayload()
→ postEngineerWarningPayload()
→ POST WARNING_APP_SCRIPT_URL
```

資料格式是 form submit，裡面帶一個 `payload` JSON 字串。

送出成功後 UI 行為：

```text
Toast：通知已送出
按鈕短暫變成：已送出
約 3 秒後恢復：發送通知
```

不新增 DB notification status，也不顯示 ack。

## 17. 完整 Dataflow 圖

```mermaid
flowchart TD
A["Browser ui/manager/index.html"] --> B["loadManagerData()"]
B --> C["fetchRealtimeDataFromDB()"]
C --> D["GET /api/manager/dashboard"]
D --> E["GetManagerDashboard()"]
E --> F["db_connection.get_connection()"]
F --> G["_build_manager_bundle_from_database()"]
G --> H["resolve_manager_date_hour()"]
H --> H1["get_manager_available_dates()"]
H --> H2["get_manager_anchor_sensor_row()"]
G --> I["get_manager_station_rows_for_hour()"]
G --> J["get_manager_station_hourly_aggregates()"]
G --> K["get_manager_sensor_rows_for_date()"]
G --> L["get_manager_distinct_batch_count_for_date()"]
G --> M["db_alert.get_unacknowledged_alerts()"]
G --> N["db_batch.get_batch_by_id()"]
H1 --> O["sensor_1min"]
H2 --> O
I --> O
J --> O
K --> O
L --> O
G --> P["intermediate bundle"]
P --> Q["build_manager_dashboard_payload()"]
Q --> R["build_manager_view()"]
Q --> S["build_manager_summary()"]
R --> T["managerView"]
S --> U["managerSummary"]
T --> V["JSON response"]
U --> V
V --> W["validateManagerDashboardPayload()"]
W --> X["applyManagerDatabaseResponse()"]
X --> Y["renderOverviewSection()"]
X --> Z["renderRecommendationSection()"]
X --> AA["renderTrendDrawer()"]
```

## 18. 通知 Dataflow 圖

```mermaid
flowchart TD
A["Click 發送通知"] --> B["sendEngineerWarningEmail()"]
B --> C["buildEngineerWarningPayload()"]
C --> D["postEngineerWarningPayload()"]
D --> E["POST WARNING_APP_SCRIPT_URL"]
E --> F["Toast: 通知已送出"]
```

## 19. API response 檢查重點

拿到 `/api/manager/dashboard` response 後，重點先看這幾個欄位：

```text
responseMeta
managerView.kpis
managerView.batchSelector
managerView.stationComparison
managerView.recommendations
managerView.trendDrawer
```

KPI 需要有：

```json
{
  "estimatedOkRatePct": 88.6,
  "estimatedNgRatePct": 11.4,
  "productionAchievementPct": 39.6,
  "dailyProduction": {
    "batchSizePcs": 264,
    "dailyTargetPcs": 20000,
    "distinctBatchCount": 30,
    "estimatedProducedPcs": 7920
  }
}
```

最重要的公式驗證：

```text
estimatedNgRatePct = 100 - estimatedOkRatePct
estimatedProducedPcs = distinctBatchCount × 264
productionAchievementPct = distinctBatchCount × 264 / 20000 × 100
```

In [ ]:
# 這個 cell 可以在實際專案電腦上使用
# 用來快速檢查 dash_latest.json 裡的 managerView 是否符合預期

import json
from pathlib import Path

path = Path("dash_latest.json")

if not path.exists():
    print("找不到 dash_latest.json")
    print("請先執行：curl.exe -s http://localhost:8011/api/manager/dashboard -o dash_latest.json")
else:
    d = json.load(open(path, encoding="utf-8"))
    manager_view = d.get("managerView", {})
    kpis = manager_view.get("kpis", {})
    daily = kpis.get("dailyProduction", {})

    print("responseMeta:")
    print(json.dumps(d.get("responseMeta", {}), ensure_ascii=False, indent=2))

    print("\nmanagerView keys:")
    print(list(manager_view.keys()))

    print("\nKPI:")
    print(json.dumps(kpis, ensure_ascii=False, indent=2))

    if daily:
        distinct_count = daily.get("distinctBatchCount")
        expected_produced = distinct_count * 264 if isinstance(distinct_count, (int, float)) else None
        expected_pct = expected_produced / 20000 * 100 if expected_produced is not None else None

        print("\nFormula check:")
        print("batchSizePcs =", daily.get("batchSizePcs"))
        print("dailyTargetPcs =", daily.get("dailyTargetPcs"))
        print("distinctBatchCount =", distinct_count)
        print("estimatedProducedPcs =", daily.get("estimatedProducedPcs"))
        print("expectedProduced =", expected_produced)
        print("productionAchievementPct =", kpis.get("productionAchievementPct"))
        print("expectedPct =", expected_pct)

## 20. 實際驗證指令

在真正跑專案的電腦上執行。

### 20.1 語法檢查

```powershell
node --check ui\manager\dashboard.js

Get-ChildItem "ui\manager\main" -Filter *.js | ForEach-Object {
  node --check $_.FullName
}

python -m unittest api.tests.test_manager_dashboard -v
```

### 20.2 啟動 Docker

```powershell
docker info
docker compose build --no-cache api frontend
.\start.ps1 -Mode start -WithData
docker compose ps
```

### 20.3 檢查 API

```powershell
curl.exe -i http://localhost:8011/api/manager/dashboard
curl.exe -s http://localhost:8011/api/manager/dashboard -o dash_latest.json
```

### 20.4 檢查 managerView

```powershell
python -c "import json; d=json.load(open('dash_latest.json',encoding='utf-8')); print('responseMeta=', d.get('responseMeta',{})); print('managerView keys=', list(d.get('managerView',{}).keys())); print(json.dumps(d.get('managerView',{}).get('kpis',{}), ensure_ascii=False, indent=2))"
```

## 21. 瀏覽器檢查清單

打開：

```text
http://localhost:8012
```

F12 → Network → 勾選 Disable cache → Ctrl + F5。

第一個 dashboard request 必須是：

```text
/api/manager/dashboard
```

不能是：

```text
/api/manager/dashboard?hour=0
```

畫面應該只看到：

```text
Manager Dashboard
Date dropdown
Hour dropdown

估算 OK rate
估算 NG rate
今日產能達成率
目前最嚴重站點

三站狀態比較表
Batch dropdown
處理建議
```

不應該看到：

```text
Source
API URL
Data window
Last updated
大量 diagnosis cards
detailed evidence
model explanation
engineer assignment
負責人
ack button
```

## 22. 最容易出錯的地方

| 問題 | 可能原因 | 檢查方式 |
|---|---|---|
| API import error | `quality_score_common.py` 沒同步 | 檢查檔案是否存在 |
| 畫面還是舊版 | frontend 沒重新 build 或瀏覽器快取 | `docker compose build --no-cache frontend`、Ctrl+F5 |
| 第一個 request 是 `?hour=0` | request builder 還有舊 bug | 檢查 `buildManagerDashboardRequestEndpoint()` |
| 今日產能達成率太大 | 用 sensor row count，不是 distinct batch | 檢查 `COUNT(DISTINCT batch_id)` |
| 尚未到站顯示 0 | 後端把 no data 當 0 | 應該回 null / `notArrived=true` |
| Batch 切換後資料沒變 | 前端沒有帶 `batch_id` 或後端沒處理 batch mode | 看 Network request URL |
| Date/Hour 切換後 Batch 還留著 | 前端沒有 reset selectedBatchId | 切 Date/Hour 後應回預設 |

## 23. 一句話總結

新版 Manager UI 的核心是：

```text
前端只打一支 GET /api/manager/dashboard，
後端把主管頁需要的資料整理成 managerView，
前端只負責顯示 KPI、三站表格、Batch dropdown、處理建議與趨勢 Drawer。
```

產量邏輯要記住：

```text
每批固定 264 pcs。
今日目標固定 20000 pcs。
batch 數永遠用 COUNT(DISTINCT batch_id)。
同一個 batch_id 只能算 1 批。
```

通知邏輯要記住：

```text
通知不是本地 API。
它是前端直接 POST 到 CONFIG.WARNING_APP_SCRIPT_URL。
```

**畫面上每一個數字到底怎麼來**。

---

# Manager Dashboard 顯示數據計算表

## 1. 畫面整體資料範圍

| 畫面位置           | 顯示內容            | 資料來源                                                                             | 怎麼決定資料範圍                                      | 注意事項                      |
| -------------- | --------------- | -------------------------------------------------------------------------------- | --------------------------------------------- | ------------------------- |
| Header         | Date dropdown   | `/api/manager/dashboard -> responseMeta.availableDates`                          | 後端從 `sensor_1min.ts` 找有資料的日期                  | 不是前端寫死                    |
| Header         | Hour dropdown   | `/api/manager/dashboard -> responseMeta.availableHours` 或 `availableHoursByDate` | 後端從 selectedDate 當天的 `sensor_1min.ts` 找有資料的小時 | 第一次載入不能帶 `hour=0`         |
| Header         | selectedDate    | `responseMeta.selectedDate`                                                      | 若前端沒指定，後端自動選 DB 最新日期                          | DB-driven                 |
| Header         | selectedHour    | `responseMeta.selectedHour`                                                      | 若前端沒指定，後端自動選 selectedDate 最新小時                | DB-driven                 |
| Batch dropdown | selectedBatchId | `managerView.batchSelector.selectedBatchId`                                      | 預設是 `null`，代表「全部批號 / 該小時累計」                   | 切 Date / Hour 後要清回 `null` |

---

## 2. 第一層 KPI：今日整體

第一層 KPI 是看 **selectedDate 今日整體**，不受 Batch dropdown 影響。

| KPI 名稱     | 畫面顯示                       | 後端欄位                                        | 資料來源                                                | 計算方式                                               | 意義                           |
| ---------- | -------------------------- | ------------------------------------------- | --------------------------------------------------- | -------------------------------------------------- | ---------------------------- |
| 估算 OK rate | `xx.x%`                    | `managerView.kpis.estimatedOkRatePct`       | selectedDate 當天所有 `sensor_1min` 感測資料                | 每筆 sensor row 用正式 quality score 公式算 OK score，最後取平均 | 這是由感測器估算的 OK 比例，不是真實檢測 OK 件數 |
| 估算 NG rate | `xx.x%`                    | `managerView.kpis.estimatedNgRatePct`       | 由 OK rate 推得                                        | `100 - estimatedOkRatePct`                         | 估算 NG 比例                     |
| 今日產能達成率    | `xx.x%`                    | `managerView.kpis.productionAchievementPct` | selectedDate 當天 `sensor_1min.batch_id`              | `COUNT(DISTINCT batch_id) × 264 / 20000 × 100%`    | 今日估算產出相對於目標 20000 pcs 的達成率   |
| 目前最嚴重站點    | `Station / Process + 主要原因` | `managerView.kpis.worstStation`             | 後端 issue / alert / recommendation / station risk 結果 | 後端從三站問題等級挑出最嚴重者                                    | 給主管快速知道最需要看的站點               |

---

## 3. OK rate 的計算來源

| 項目         | 說明                                                                         |
| ---------- | -------------------------------------------------------------------------- |
| 資料表        | `sensor_1min`                                                              |
| 使用欄位       | `paint_flow_ml_min`, `air_pressure_bar`, `spray_width_mm`, `path_error_mm` |
| 後端公式來源     | `services/integrated_service/quality_score_common.py`                      |
| 原始正式邏輯     | `_quality_score_from_formal_row(row, station_id)`                          |
| 計算方式       | 每筆感測資料依噴塗流量、空壓、噴幅、路徑誤差扣分後得到 quality score                                  |
| OK rate 意義 | 感測條件越接近正常範圍，估算 OK rate 越高                                                  |
| NG rate 意義 | `100 - OK rate`                                                            |
| 重要限制       | 這不是人工品檢結果，也不是真實 OK/NG 件數，是 sensor-weighted estimated rate                  |

---

## 4. 今日產能達成率計算

| 項目         | 數值 / 公式                        |
| ---------- | ------------------------------ |
| 每批固定數量     | `264 pcs`                      |
| 今日目標       | `20000 pcs`                    |
| 今日 batch 數 | `COUNT(DISTINCT batch_id)`     |
| 今日估算產出 pcs | `今日 distinct batch_id 數 × 264` |
| 今日產能達成率    | `今日估算產出 pcs / 20000 × 100%`    |

範例：

| 項目                     |                           數值 |
| ---------------------- | ---------------------------: |
| 今日 distinct batch_id 數 |                         30 批 |
| 每批 pcs                 |                      264 pcs |
| 今日估算產出                 |        `30 × 264 = 7920 pcs` |
| 今日目標                   |                    20000 pcs |
| 今日產能達成率                | `7920 / 20000 × 100 = 39.6%` |

最重要規則：

```text
同一個 batch_id 只能算 1 批。
不能用 sensor_1min row count。
不能用 station-batch 組合數。
```

---

## 5. Batch dropdown

| 顯示項目         | 後端欄位               | 資料來源                                                 | 計算 / 取得方式                                               | 注意事項                   |
| ------------ | ------------------ | ---------------------------------------------------- | ------------------------------------------------------- | ---------------------- |
| 全部批號 / 該小時累計 | `defaultModeLabel` | 固定文字                                                 | 預設選項                                                    | selectedBatchId = null |
| Batch 清單     | `availableBatches` | selectedDate + selectedHour 的 `sensor_1min.batch_id` | `COUNT(DISTINCT batch_id)` 或 `SELECT DISTINCT batch_id` | 同一批多筆 sensor row 只出現一次 |
| 問題標籤         | `issueLabel`       | 後端問題判斷結果                                             | NG rate、NG pcs 上升、稼動率、active alert 等條件判斷                | 用中文短句，例如 `NG 率偏高`      |
| 顯示文字         | `displayLabel`     | batchId + issueLabel                                 | 例如 `BATCH-002 ⚠ NG 率偏高`                                 | 不用英文                   |

---

## 6. 三站狀態比較表

三站表格會跟著 Batch dropdown 模式切換。

| 欄位                | 後端欄位                         | Batch 未選時計算                                   | Batch 已選時計算                                           | 注意事項             |
| ----------------- | ---------------------------- | --------------------------------------------- | ----------------------------------------------------- | ---------------- |
| Station / Process | `stationName`, `processName` | 固定三站資料                                        | 固定三站資料                                                | Station row 永遠顯示 |
| 稼動率 %             | `utilizationPct`             | selectedDate + selectedHour + station 的該小時稼動率 | selectedDate + selectedHour + batch_id + station 的稼動率 | 沒資料顯示 `—`        |
| 預估 OK pcs         | `estimatedOkPcs`             | `該小時該站 distinct batch 數 × 264 × OK rate`      | `264 × 該 batch 該 station OK rate`                     | 是估算，不是真實 OK 件數   |
| 預估 NG pcs         | `estimatedNgPcs`             | `該小時該站 distinct batch 數 × 264 × NG rate`      | `264 × 該 batch 該 station NG rate`                     | 是估算，不是真實 NG 件數   |
| 主要問題              | `mainIssue`                  | 後端依該站該小時問題判斷                                  | 後端依該 batch 該站問題判斷                                     | 尚未到站顯示 `尚未到站`    |

---

## 7. 三站表格：Batch 未選模式

模式：

```text
全部批號 / 該小時累計
```

| 顯示數據       | 計算方式                                                                             |
| ---------- | -------------------------------------------------------------------------------- |
| 該站總 pcs    | `selectedDate + selectedHour + station_id` 範圍內的 `COUNT(DISTINCT batch_id) × 264` |
| 該站 OK rate | 該小時該站 sensor rows 算出平均 OK rate                                                   |
| 該站 NG rate | `100 - OK rate`                                                                  |
| 預估 OK pcs  | `該站總 pcs × OK rate / 100`                                                        |
| 預估 NG pcs  | `該站總 pcs × NG rate / 100`                                                        |

範例：

| 項目                             |                    數值 |
| ------------------------------ | --------------------: |
| Station 1 該小時 distinct batch 數 |                   3 批 |
| 每批 pcs                         |               264 pcs |
| Station 1 該小時總 pcs             |   `3 × 264 = 792 pcs` |
| OK rate                        |                   90% |
| NG rate                        |                   10% |
| 預估 OK pcs                      | `792 × 90% = 713 pcs` |
| 預估 NG pcs                      |  `792 × 10% = 79 pcs` |

---

## 8. 三站表格：Batch 已選模式

模式：

```text
BATCH-xxx
```

| 顯示數據       | 計算方式                                                                      |
| ---------- | ------------------------------------------------------------------------- |
| 該批總 pcs    | 固定 `264 pcs`                                                              |
| 該站 OK rate | selectedDate + selectedHour + selectedBatchId + station 的 sensor rows 算平均 |
| 該站 NG rate | `100 - OK rate`                                                           |
| 預估 OK pcs  | `264 × OK rate / 100`                                                     |
| 預估 NG pcs  | `264 × NG rate / 100`                                                     |

範例：

| 項目                |                    數值 |
| ----------------- | --------------------: |
| Batch 總 pcs       |               264 pcs |
| Station 1 OK rate |                   91% |
| Station 1 NG rate |                    9% |
| 預估 OK pcs         | `264 × 91% = 240 pcs` |
| 預估 NG pcs         |   `264 × 9% = 24 pcs` |

如果該 Batch 尚未到某站：

| 欄位        | 顯示     |
| --------- | ------ |
| 稼動率       | `—`    |
| 預估 OK pcs | `—`    |
| 預估 NG pcs | `—`    |
| 主要問題      | `尚未到站` |

不要顯示：

```text
0 pcs
0%
No issue
```

---

## 9. 右側 Drawer 趨勢圖

| Drawer 狀態 | 顯示資料                      | 資料來源                        | 計算方式                             |
| --------- | ------------------------- | --------------------------- | -------------------------------- |
| Batch 未選  | 該 station 今日每小時全部批號趨勢     | `managerView.trendDrawer`   | 每小時依 station 聚合所有 distinct batch |
| Batch 已選  | 該 station 今日每小時該 batch 趨勢 | `managerView.trendDrawer`   | 每小時依 station + batch_id 聚合       |
| 稼動率趨勢     | 00:00~23:00 每小時稼動率        | hourly aggregates           | 每小時 utilization                  |
| OK pcs 趨勢 | 00:00~23:00 每小時預估 OK pcs  | hourly aggregates + OK rate | `hour_total_pcs × OK rate`       |
| NG pcs 趨勢 | 00:00~23:00 每小時預估 NG pcs  | hourly aggregates + NG rate | `hour_total_pcs × NG rate`       |

---

## 10. Drawer 摘要

| 模式               | 摘要顯示      | 計算方式                              |
| ---------------- | --------- | --------------------------------- |
| Batch 未選         | 今日累計      | 該 station 今日 00~23 每小時數值加總        |
| Batch 未選         | 本小時       | selectedHour 的數值                  |
| Batch 未選         | 最高小時      | 今日 24 小時中最大值                      |
| Batch 已選         | Batch     | selectedBatchId                   |
| Batch 已選         | 此批總數      | 固定 `264 pcs`                      |
| Batch 已選點 OK pcs | 預估 OK pcs | `264 × 該 batch 該 station OK rate` |
| Batch 已選點 NG pcs | 預估 NG pcs | `264 × 該 batch 該 station NG rate` |
| Batch 已選點稼動率     | 該站稼動率     | 該 batch 該 station utilization     |

---

## 11. 處理建議

| 顯示項目              | 後端欄位                         | 資料來源                           | 判斷方式                               |
| ----------------- | ---------------------------- | ------------------------------ | ---------------------------------- |
| Station / Process | `stationName`, `processName` | station metadata / station row | 後端整理                               |
| 問題                | `mainIssue`                  | 後端 issue 判斷                    | NG rate、NG pcs 上升、稼動率、active alert |
| 建議                | `recommendation`             | 後端 recommendation              | 依 mainIssue 產生建議                   |
| 等級                | `level`                      | 後端判斷                           | warning / alarm / critical         |
| 發送通知              | button                       | 前端 render                      | 點擊後送 Apps Script                   |

---

## 12. 問題判斷條件

| 條件                                    | 結果      |
| ------------------------------------- | ------- |
| `status = warning / alarm / critical` | 列入處理建議  |
| 估算 NG rate >= 5%                      | warning |
| 估算 NG rate >= 10%                     | alarm   |
| 本小時 NG pcs >= 今日每小時平均 NG pcs × 1.5    | warning |
| 本小時 NG pcs >= 今日每小時平均 NG pcs × 2.0    | alarm   |
| 今日平均稼動率 < 70%                         | warning |
| 今日平均稼動率 < 50%                         | alarm   |
| 有 active alert / recommendation       | 列入處理建議  |

如果沒有任何問題：

| 區塊   | 顯示         |
| ---- | ---------- |
| 處理建議 | `目前無需處理事項` |

---

## 13. 發送通知資料

| 通知欄位    | 來源                                   |
| ------- | ------------------------------------ |
| Date    | `responseMeta.selectedDate`          |
| Hour    | `responseMeta.selectedHour`          |
| Batch   | selectedBatchId，沒選則顯示 `全部批號 / 該小時累計` |
| Station | recommendation 的 stationName         |
| Process | recommendation 的 processName         |
| 問題      | recommendation 的 mainIssue           |
| 建議      | recommendation 的 recommendation      |
| 等級      | recommendation 的 level               |
| 收件者     | 現有 engineer email / config           |
| 送出端點    | `CONFIG.WARNING_APP_SCRIPT_URL`      |

通知不是本地 API Server route。
是前端直接送外部 Apps Script。

---

# 最重要結論

| 顯示數據    | 是真實值還是估算值  | 說明                             |
| ------- | ---------- | ------------------------------ |
| OK rate | 估算值        | 由 sensor_1min 感測狀態推估           |
| NG rate | 估算值        | `100 - OK rate`                |
| OK pcs  | 估算值        | 由固定批量 264 pcs × OK rate 推估     |
| NG pcs  | 估算值        | 由固定批量 264 pcs × NG rate 推估     |
| 今日產能達成率 | 估算值        | distinct batch 數 × 264 / 20000 |
| Batch 數 | DB 統計值     | `COUNT(DISTINCT batch_id)`     |
| 稼動率     | DB / 後端彙總值 | 後端依 sensor/hour/station 聚合     |
| 主要問題    | 後端判斷值      | 根據 NG rate、NG pcs、稼動率、alert 判斷 |
| 通知狀態    | 前端暫時 UI 狀態 | 不寫入 DB                         |

這張表要記住一句話：

```text
Manager Dashboard 目前不是顯示真實品檢 OK/NG 件數，
而是用 sensor_1min 感測資料估算 OK/NG rate，
再用固定每批 264 pcs 推估 OK/NG pcs。
```
